In [ ]:
#!/usr/bin/python3
# -*- coding: utf-8 -*-
"""
使用 DuckDB 查询按 year/month/htsc_code 分区的 Parquet 数据
数据根目录: D:\database\stock_basic_data
"""

import duckdb
import os
from datetime import datetime

# ==================== 配置 ====================
BASE_PATH = r"D:\database\stock_basic_data"

# ==================== DuckDB 查询函数 ====================
def query_stock_data(year: int, month: int, htsc_code: str = None, limit: int = None):
    """
    查询指定年、月、股票代码的数据
    
    参数:
        year: 年份 (例如 2026)
        month: 月份 (1-12)
        htsc_code: 股票代码 (例如 "600519.SH")，如果不填则查询该月所有股票
        limit: 限制返回行数（调试用）
    """
    conn = duckdb.connect(database=":memory:")
    
    # 构建分区路径
    month_str = f"{month:02d}"
    partition_path = f"{BASE_PATH}/year={year}/month={month_str}"
    
    if not os.path.exists(partition_path):
        print(f"错误：路径不存在 → {partition_path}")
        return None
    
    # 构建 SQL
    if htsc_code:
        sql = f"""
        SELECT * FROM read_parquet('{partition_path}/htsc_code={htsc_code}/*.parquet')
        """
        if limit:
            sql += f" LIMIT {limit}"
    else:
        sql = f"""
        SELECT * FROM read_parquet('{partition_path}/**/*.parquet')
        """
        if limit:
            sql += f" LIMIT {limit}"
    
    print(f"正在查询: 年={year}, 月={month}, 股票={htsc_code or '全部'}")
    print(f"查询路径: {partition_path}")
    
    try:
        df = conn.sql(sql).df()   # 返回 pandas DataFrame
        print(f"查询成功！共 {len(df)} 条记录")
        return df
    except Exception as e:
        print(f"查询失败: {e}")
        return None


# ==================== 使用示例 ====================

if __name__ == "__main__":
    print("=== DuckDB 分区查询工具 ===")
    print(f"数据根目录: {BASE_PATH}\n")
    
    # 示例1：查询某只股票某个月的数据
    df1 = query_stock_data(year=2026, month=2, htsc_code="600519.SH")
    if df1 is not None:
        print("\n前5条数据预览:")
        print(df1.head())
    
    # 示例2：查询某个月所有股票（数据量会很大，建议加 limit）
    # df2 = query_stock_data(year=2026, month=2, limit=1000)
    
    # 示例3：查询多个月份（可自行扩展）
    # for m in [1,2,3]:
    #     df = query_stock_data(2026, m, "600519.SH")